# FAR tests using different datasets

In [ ]:
import sys

!{sys.executable} -m pip install shap
!{sys.executable} -m pip install nbimporter

In [2]:
import numpy as np
import pandas as pd
import sklearn as skl
import zipfile as zf
import matplotlib.pyplot as plt
import nbimporter
import shap

import Ft_Att_Rank as far

# Iris dataset

In [3]:
import sklearn.datasets

iris= sklearn.datasets.load_iris()

X_iris= iris.data
Y_iris= iris.target

In [4]:
# convert iris to a df and drop the setosa rows
df_iris= pd.DataFrame(data= np.c_[X_iris, Y_iris], columns= iris['feature_names'] + ['target'])
df_iris= df_iris[df_iris['target']!= 0].reset_index(drop= True)

In [ ]:
# split df_iris into features (x) and target (y)
df_iris_x= df_iris.loc[:,df_iris.columns[0:4]]
df_iris_y= df_iris.loc[:,df_iris.columns[4:5]]

df_iris_x.shape

In [ ]:
# ML model - Random forest
import sklearn.ensemble

train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(df_iris_x,df_iris_y,train_size=0.80,random_state=1234)

rf= sklearn.ensemble.RandomForestClassifier(n_estimators=500,n_jobs=2)
rf.fit(train, labels_train.values.ravel())
rf_acc= sklearn.metrics.accuracy_score(labels_test, rf.predict(test))

rf_acc

In [ ]:
# ML model - XGBoost random forest
import xgboost as xgb

xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',verbosity=0)
xgb_model.fit(train, labels_train.values.ravel())
xgb_acc= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

xgb_acc

In [ ]:
# Feature Attribution using Raking
repeat_train= 1
num_fts= len(train.columns)

acc_all_fts= far.train_model_get_acc_mean(xgb_model, train, test, labels_train.values.ravel(), 
                                          labels_test.values.ravel(), repeat_train)

replace_ft= far.replace_values(train,train.columns,num_type='mean')

acc_no_i, acc_no_ij= far.remove_and_retrain_v1(xgb_model, replace_ft, train, test, labels_train.values.ravel(),
                                               labels_test.values.ravel(), repeat_train)

In [ ]:
# get the probability matrix
p_matrix1= far.get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= far.get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= far.get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= far.get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# get the stationary distribution
stationary_d1= far.p_matrix_to_stationary_dist(p_matrix1)
stationary_d2= far.p_matrix_to_stationary_dist(p_matrix2)
stationary_d3= far.p_matrix_to_stationary_dist(p_matrix3)
stationary_d4= far.p_matrix_to_stationary_dist(p_matrix4)

In [ ]:
ft_ranking= pd.Series(stationary_d1, index=train.columns).sort_values(ascending=False)
ft_ranking

In [ ]:
ft_ranking= pd.Series(stationary_d2, index=train.columns).sort_values(ascending=False)
ft_ranking

In [ ]:
ft_ranking= pd.Series(stationary_d3, index=train.columns).sort_values(ascending=False)
ft_ranking

In [ ]:
ft_ranking= pd.Series(stationary_d4, index=train.columns).sort_values(ascending=False)
ft_ranking

# Wine dataset

In [ ]:
wine= pd.read_csv('datasets/wine.data',header=None)

wine.columns= ['target','alcohol','malicAcid','ash','ashalcalinity','magnesium','totalPhenols','flavanoids','nonFlavanoidPhenols','proanthocyanins',
               'colorIntensity','hue','od280_od315','proline']

wine= wine[wine['target']!= 3].reset_index(drop= True)

x_wine= wine.iloc[:,1:len(wine.columns)].copy()
y_wine= np.asarray(wine['target'])

x_wine.shape

In [ ]:
# ML model - XGBoost random forest
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(x_wine,y_wine,train_size=0.80,random_state=1234)

xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss')
xgb_model.fit(train, labels_train)
xgb_acc= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

xgb_acc

In [ ]:
# Feature Attribution using Raking
repeat_train= 1
num_fts= len(train.columns)

acc_all_fts= far.train_model_get_acc_mean(xgb_model, train, test, labels_train, labels_test, repeat_train)

replace_ft= far.replace_values(train,train.columns,num_type='mean')

acc_no_i, acc_no_ij= far.remove_and_retrain_v1(xgb_model, replace_ft, train, test, labels_train, labels_test, repeat_train)

In [ ]:
# get the probability matrix
p_matrix1= far.get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= far.get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= far.get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= far.get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# get the stationary distribution
stationary_d1= far.p_matrix_to_stationary_dist(p_matrix1)
stationary_d2= far.p_matrix_to_stationary_dist(p_matrix2)
stationary_d3= far.p_matrix_to_stationary_dist(p_matrix3)
stationary_d4= far.p_matrix_to_stationary_dist(p_matrix4)

In [ ]:
ft_ranking= pd.Series(stationary_d1, index=train.columns).sort_values(ascending=False)
ft_ranking

In [ ]:
ft_ranking= pd.Series(stationary_d2, index=train.columns).sort_values(ascending=False)
ft_ranking

In [ ]:
ft_ranking= pd.Series(stationary_d3, index=train.columns).sort_values(ascending=False)
ft_ranking

In [ ]:
ft_ranking= pd.Series(stationary_d4, index=train.columns).sort_values(ascending=False)
ft_ranking

# Titanic dataset

In [ ]:
# https://www.kaggle.com/c/titanic
ds= zf.ZipFile('datasets/titanic.zip')

train_data= pd.read_csv(ds.open('train.csv'))
test_data= pd.read_csv(ds.open('test.csv'))

X_all= pd.concat([train_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']],
                   test_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']]]).set_index('PassengerId')

y_train= train_data[['PassengerId','Survived']].set_index('PassengerId')['Survived']

numeric_columns= ['Age','SibSp','Parch','Fare']
categor_columns= list(filter(lambda x:x not in numeric_columns,X_all.columns))

X_train= X_all.iloc[:len(train_data),:].copy()
X_test= X_all.iloc[len(train_data):].copy()

In [ ]:
X_train= far.pre_proc_fillna_num_fts(X_train,numeric_columns,num_type='median')
X_train= far.pre_proc_fillna_cat_fts(X_train,categor_columns,cat_type='mode')

X_train_ohe= pd.get_dummies(X_train,columns=categor_columns)

X_train.shape

In [ ]:
# ML model - XGBoost random forest
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(X_train_ohe,y_train,train_size=0.80,random_state=1234)

xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',use_label_encoder=False)
xgb_model.fit(train, labels_train.values.ravel())
xgb_acc= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

xgb_acc

In [ ]:
# Feature Attribution using Raking
repeat_train= 1
num_fts= len(train.columns)

acc_all_fts= far.train_model_get_acc_mean(xgb_model, train, test, labels_train, labels_test, repeat_train)

replace_ft= far.replace_values(train,numeric_columns,num_type='mean',cat_type='median')

acc_no_i, acc_no_ij= far.remove_and_retrain_v1(xgb_model, replace_ft, train, test, labels_train, labels_test, repeat_train)

In [ ]:
# get the probability matrix
p_matrix1= far.get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= far.get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= far.get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= far.get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# get the stationary distribution
stationary_d1= far.p_matrix_to_stationary_dist(p_matrix1)
stationary_d2= far.p_matrix_to_stationary_dist(p_matrix2)
stationary_d3= far.p_matrix_to_stationary_dist(p_matrix3)
stationary_d4= far.p_matrix_to_stationary_dist(p_matrix4)

In [ ]:
rp_list= {'Sex_male':'Sex','Sex_female':'Sex','Pclass_1':'Pclass','Pclass_2':'Pclass','Pclass_3':'Pclass',
          'Embarked_S':'Embarked','Embarked_Q':'Embarked','Embarked_C':'Embarked'}

In [ ]:
ft_ranking= pd.Series(stationary_d1, index=train.columns).sort_values(ascending=False)
ft_ranking

In [ ]:
far.ft_importance_df(stationary_d1,train.columns,rp_list)

In [ ]:
ft_ranking= pd.Series(stationary_d2, index=train.columns).sort_values(ascending=False)
ft_ranking

In [ ]:
far.ft_importance_df(stationary_d2,train.columns,rp_list)

In [ ]:
ft_ranking= pd.Series(stationary_d3, index=train.columns).sort_values(ascending=False)
ft_ranking

In [ ]:
far.ft_importance_df(stationary_d2,train.columns,rp_list)

In [ ]:
ft_ranking= pd.Series(stationary_d4, index=train.columns).sort_values(ascending=False)
ft_ranking

In [ ]:
far.ft_importance_df(stationary_d4,train.columns,rp_list)

# Heartrisk dataset

In [ ]:
# https://www.kaggle.com/pritsheta/heart-attack
ds= zf.ZipFile('datasets/heart.zip')

heart_data= pd.read_csv(ds.open('heart.csv'))

x_heart= heart_data.iloc[:,:(len(heart_data.columns)-1)].copy()
y_heart= np.asarray(heart_data['target'])

x_heart.shape

In [ ]:
# ML model - XGBoost random forest
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(x_heart,y_heart,train_size=0.80,random_state=1234)

xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',use_label_encoder=False)
xgb_model.fit(train, labels_train)
xgb_acc= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

xgb_acc

In [ ]:
# Feature Attribution using Raking
repeat_train= 1
num_fts= len(train.columns)

acc_all_fts= far.train_model_get_acc_mean(xgb_model, train, test, labels_train, labels_test, repeat_train)

replace_ft= far.replace_values(train,train.columns,num_type='mean')

acc_no_i, acc_no_ij= far.remove_and_retrain_v1(xgb_model, replace_ft, train, test, labels_train, labels_test, repeat_train)

In [ ]:
# get the probability matrix
p_matrix1= far.get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= far.get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= far.get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= far.get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# get the stationary distribution
stationary_d1= far.p_matrix_to_stationary_dist(p_matrix1)
stationary_d2= far.p_matrix_to_stationary_dist(p_matrix2)
stationary_d3= far.p_matrix_to_stationary_dist(p_matrix3)
stationary_d4= far.p_matrix_to_stationary_dist(p_matrix4)

In [ ]:
ft_ranking= pd.Series(stationary_d1, index=train.columns).sort_values(ascending=False)
ft_ranking

In [ ]:
ft_ranking= pd.Series(stationary_d2, index=train.columns).sort_values(ascending=False)
ft_ranking

In [ ]:
ft_ranking= pd.Series(stationary_d3, index=train.columns).sort_values(ascending=False)
ft_ranking

In [ ]:
ft_ranking= pd.Series(stationary_d4, index=train.columns).sort_values(ascending=False)
ft_ranking

# Breast cancer Wisconsin dataset

In [ ]:
# https://www.kaggle.com/uciml/breast-cancer-wisconsin-data
wdbc= sklearn.datasets.load_breast_cancer()

X_wb_cancer= pd.DataFrame(wdbc.data,columns=wdbc.feature_names)
Y_wb_cancer= wdbc.target

X_wb_cancer.shape

In [48]:
# ML model - XGBoost random forest
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(X_wb_cancer,Y_wb_cancer,train_size=0.80,random_state=1234)

xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',use_label_encoder=False)
xgb_model.fit(train, labels_train)
xgb_acc= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

xgb_acc

0.9473684210526315

In [49]:
# Feature Attribution using Raking
repeat_train= 1
num_fts= len(train.columns)

acc_all_fts= far.train_model_get_acc_mean(xgb_model, train, test, labels_train, labels_test, repeat_train)

replace_ft= far.replace_values(train,train.columns,num_type='mean')

acc_no_i, acc_no_ij= far.remove_and_retrain_v1(xgb_model, replace_ft, train, test, labels_train, labels_test, repeat_train)

In [50]:
# get the probability matrix
p_matrix1= far.get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= far.get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= far.get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= far.get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# get the stationary distribution
stationary_d1= far.p_matrix_to_stationary_dist(p_matrix1)
stationary_d2= far.p_matrix_to_stationary_dist(p_matrix2)
stationary_d3= far.p_matrix_to_stationary_dist(p_matrix3)
stationary_d4= far.p_matrix_to_stationary_dist(p_matrix4)

In [51]:
ft_ranking= pd.Series(stationary_d1, index=train.columns).sort_values(ascending=False)
ft_ranking

worst texture              1.450848e-01
mean texture               1.106096e-01
worst area                 9.071237e-02
worst concavity            8.893382e-02
worst perimeter            7.644444e-02
area error                 6.932635e-02
worst radius               6.771069e-02
worst concave points       6.607988e-02
mean concave points        5.593596e-02
mean smoothness            3.787910e-02
mean area                  2.887780e-02
mean concavity             2.168325e-02
mean radius                2.109623e-02
worst smoothness           2.063137e-02
worst fractal dimension    1.600510e-02
concavity error            1.005542e-02
mean compactness           8.904488e-03
radius error               8.082448e-03
worst compactness          7.257570e-03
symmetry error             7.257570e-03
worst symmetry             7.257570e-03
mean perimeter             7.121696e-03
texture error              5.949683e-03
perimeter error            5.426599e-03
mean fractal dimension     5.124805e-03


In [52]:
ft_ranking= pd.Series(stationary_d2, index=train.columns).sort_values(ascending=False)
ft_ranking

mean smoothness            0.122922
worst area                 0.115768
worst concave points       0.094101
mean concave points        0.056460
mean texture               0.044780
worst texture              0.039820
worst concavity            0.036076
mean radius                0.030334
worst compactness          0.028472
radius error               0.026622
area error                 0.026114
mean perimeter             0.026019
mean concavity             0.025093
fractal dimension error    0.024860
worst perimeter            0.023453
concavity error            0.022829
worst radius               0.021633
mean symmetry              0.021302
worst symmetry             0.021233
symmetry error             0.021233
perimeter error            0.021006
worst fractal dimension    0.020746
texture error              0.019944
smoothness error           0.018402
compactness error          0.018169
concave points error       0.018169
mean compactness           0.017386
mean fractal dimension     0

In [53]:
ft_ranking= pd.Series(stationary_d3, index=train.columns).sort_values(ascending=False)
ft_ranking

mean smoothness            0.125663
worst area                 0.108075
worst concave points       0.093324
mean concave points        0.057461
mean texture               0.048207
worst texture              0.037662
worst concavity            0.034261
mean radius                0.030394
worst radius               0.027785
area error                 0.027288
mean perimeter             0.025921
worst perimeter            0.025211
mean concavity             0.024528
worst compactness          0.023533
radius error               0.022780
fractal dimension error    0.021840
concavity error            0.020715
perimeter error            0.020438
worst fractal dimension    0.020259
symmetry error             0.020140
worst symmetry             0.020140
texture error              0.019835
mean symmetry              0.019624
compactness error          0.018889
concave points error       0.018889
mean compactness           0.018741
smoothness error           0.018592
mean fractal dimension     0

In [54]:
ft_ranking= pd.Series(stationary_d4, index=train.columns).sort_values(ascending=False)
ft_ranking

worst texture              1.268658e-01
area error                 1.223114e-01
mean texture               1.091927e-01
worst radius               9.205973e-02
mean smoothness            8.425856e-02
worst area                 7.758575e-02
worst concave points       7.541681e-02
mean concave points        6.409379e-02
mean area                  4.970459e-02
worst perimeter            4.741159e-02
worst concavity            4.530037e-02
worst smoothness           1.647827e-02
mean radius                1.188608e-02
mean concavity             1.175112e-02
worst fractal dimension    9.264057e-03
mean compactness           6.667604e-03
mean perimeter             5.074544e-03
texture error              4.714359e-03
mean fractal dimension     4.625873e-03
concavity error            4.549697e-03
radius error               4.415574e-03
perimeter error            4.382246e-03
worst compactness          4.327088e-03
worst symmetry             4.327088e-03
symmetry error             4.327088e-03


# Heart failure prediction dataset

In [55]:
# https://www.kaggle.com/fedesoriano/heart-failure-prediction
ds= zf.ZipFile('datasets/heart_2.zip')

heart_2_data= pd.read_csv(ds.open('heart.csv'))

x_heart2= heart_2_data.iloc[:,:(len(heart_2_data.columns)-1)].copy()
y_heart2= np.asarray(heart_2_data['HeartDisease'])

x_heart2.shape

(918, 11)

In [56]:
numeric_columns= ['Age','RestingBP','Cholesterol','MaxHR','Oldpeak']
categor_columns= list(filter(lambda x:x not in numeric_columns,x_heart2.columns))

Sex= [1 if s == 'M' else 0 for s in x_heart2['Sex']]
ExerciseAngina= [1 if e == 'Y' else 0 for e in x_heart2['ExerciseAngina']]

x_heart2['Sex']= Sex
x_heart2['ExerciseAngina']= ExerciseAngina

x_heart2_ohe= pd.get_dummies(x_heart2,columns=['ChestPainType','RestingECG','ST_Slope'])

x_heart2_ohe.shape

(918, 18)

In [57]:
# ML model - XGBoost random forest
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(x_heart2_ohe,y_heart2,train_size=0.80,random_state=1234)

xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',use_label_encoder=False)
xgb_model.fit(train, labels_train)
xgb_acc= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

xgb_acc

0.8913043478260869

In [58]:
# Feature Attribution using Raking
repeat_train= 1
num_fts= len(train.columns)

acc_all_fts= far.train_model_get_acc_mean(xgb_model, train, test, labels_train, labels_test, repeat_train)

replace_ft= far.replace_values(train,train.columns,num_type='mean')

acc_no_i, acc_no_ij= far.remove_and_retrain_v1(xgb_model, replace_ft, train, test, labels_train, labels_test, repeat_train)

In [59]:
# get the probability matrix
p_matrix1= far.get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= far.get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= far.get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= far.get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# get the stationary distribution
stationary_d1= far.p_matrix_to_stationary_dist(p_matrix1)
stationary_d2= far.p_matrix_to_stationary_dist(p_matrix2)
stationary_d3= far.p_matrix_to_stationary_dist(p_matrix3)
stationary_d4= far.p_matrix_to_stationary_dist(p_matrix4)

In [60]:
rp_list= {'ChestPainType_ASY':'ChestPainType','ChestPainType_ATA':'ChestPainType','ChestPainType_NAP':'ChestPainType','ChestPainType_TA':'ChestPainType',
          'RestingECG_LVH':'RestingECG','RestingECG_Normal':'RestingECG','RestingECG_ST':'RestingECG',
          'ST_Slope_Down':'ST_Slope','ST_Slope_Flat':'ST_Slope','ST_Slope_Up':'ST_Slope'}

In [61]:
ft_ranking= pd.Series(stationary_d1, index=train.columns).sort_values(ascending=False)
ft_ranking

Oldpeak              1.324297e-01
ST_Slope_Up          1.294279e-01
Cholesterol          1.207097e-01
ChestPainType_ASY    1.117173e-01
RestingBP            9.852778e-02
ST_Slope_Flat        8.554786e-02
ExerciseAngina       7.177764e-02
MaxHR                5.348612e-02
FastingBS            4.710356e-02
Sex                  4.678875e-02
Age                  4.211399e-02
RestingECG_LVH       3.528255e-02
RestingECG_Normal    1.227630e-02
ChestPainType_ATA    6.626210e-03
ChestPainType_TA     3.498051e-03
ChestPainType_NAP    2.600022e-03
RestingECG_ST        8.666738e-05
ST_Slope_Down        4.936462e-17
dtype: float64

In [62]:
far.ft_importance_df(stationary_d1,train.columns,rp_list)

,Feature,Importance
0,ST_Slope,0.214976
1,Oldpeak,0.132430
2,ChestPainType,0.124442
3,Cholesterol,0.120710
4,RestingBP,0.098528
5,ExerciseAngina,0.071778
6,MaxHR,0.053486
7,RestingECG,0.047646
8,FastingBS,0.047104
9,Sex,0.046789


In [63]:
ft_ranking= pd.Series(stationary_d2, index=train.columns).sort_values(ascending=False)
ft_ranking

ST_Slope_Up          0.134080
ChestPainType_NAP    0.096787
ChestPainType_ASY    0.080382
ExerciseAngina       0.077604
ST_Slope_Flat        0.073980
ChestPainType_TA     0.065683
ST_Slope_Down        0.065022
FastingBS            0.055817
MaxHR                0.043516
RestingECG_LVH       0.041438
RestingECG_Normal    0.040374
Oldpeak              0.037981
RestingECG_ST        0.035768
Cholesterol          0.035354
ChestPainType_ATA    0.035102
RestingBP            0.030720
Sex                  0.025854
Age                  0.024290
dtype: float64

In [64]:
far.ft_importance_df(stationary_d2,train.columns,rp_list)

,Feature,Importance
0,ChestPainType,0.277953
1,ST_Slope,0.273083
2,RestingECG,0.117580
3,ExerciseAngina,0.077604
4,FastingBS,0.055817
5,MaxHR,0.043516
6,Oldpeak,0.037981
7,Cholesterol,0.035354
8,RestingBP,0.030720
9,Sex,0.025854


In [65]:
ft_ranking= pd.Series(stationary_d3, index=train.columns).sort_values(ascending=False)
ft_ranking

ChestPainType_NAP    0.107558
ST_Slope_Up          0.106056
ChestPainType_ASY    0.088187
ST_Slope_Flat        0.063925
ExerciseAngina       0.063729
FastingBS            0.061730
MaxHR                0.046565
Oldpeak              0.046023
ChestPainType_TA     0.045024
ST_Slope_Down        0.044266
RestingECG_LVH       0.043770
RestingECG_Normal    0.043210
RestingBP            0.042880
Cholesterol          0.042030
RestingECG_ST        0.041362
ChestPainType_ATA    0.040912
Sex                  0.036499
Age                  0.036156
dtype: float64

In [66]:
far.ft_importance_df(stationary_d3,train.columns,rp_list)

,Feature,Importance
0,ChestPainType,0.281682
1,ST_Slope,0.214247
2,RestingECG,0.128343
3,ExerciseAngina,0.063729
4,FastingBS,0.061730
5,MaxHR,0.046565
6,Oldpeak,0.046023
7,RestingBP,0.042880
8,Cholesterol,0.042030
9,Sex,0.036499


In [67]:
ft_ranking= pd.Series(stationary_d4, index=train.columns).sort_values(ascending=False)
ft_ranking

Oldpeak              1.729517e-01
ChestPainType_ASY    1.378086e-01
Cholesterol          1.262387e-01
ExerciseAngina       9.131846e-02
RestingBP            8.993494e-02
ST_Slope_Up          8.315521e-02
MaxHR                6.449903e-02
FastingBS            6.073327e-02
ST_Slope_Flat        5.439405e-02
Age                  4.002266e-02
Sex                  2.357352e-02
ChestPainType_NAP    2.176364e-02
RestingECG_LVH       1.964042e-02
RestingECG_Normal    7.899765e-03
ChestPainType_ATA    4.411573e-03
ChestPainType_TA     1.279311e-03
RestingECG_ST        3.752353e-04
ST_Slope_Down       -5.758602e-17
dtype: float64

In [68]:
far.ft_importance_df(stationary_d4,train.columns,rp_list)

,Feature,Importance
0,Oldpeak,0.172952
1,ChestPainType,0.165263
2,ST_Slope,0.137549
3,Cholesterol,0.126239
4,ExerciseAngina,0.091318
5,RestingBP,0.089935
6,MaxHR,0.064499
7,FastingBS,0.060733
8,Age,0.040023
9,RestingECG,0.027915
